In [1]:
import pandas as pd
import joblib
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_pickle("../data/processed/movies_clustered.pkl")
tfidf_matrix = sparse.load_npz("../models/tfidf_matrix.npz")

df.shape, tfidf_matrix.shape

((1719, 22), (1719, 5000))

In [2]:
def recommend(title, n=5):
    matches = df.index[df["title"].str.lower() == title.lower()]
    if len(matches) == 0:
        print(f"Movie '{title}' not found.")
        return None

    idx = matches[0]
    movie_vector = tfidf_matrix[idx]

    sims = cosine_similarity(movie_vector, tfidf_matrix).flatten()

    similar_idx = sims.argsort()[::-1]
    similar_idx = [i for i in similar_idx if i != idx][:n]

    results = df.iloc[similar_idx][["title", "genres", "year"]].copy()
    results["similarity"] = sims[similar_idx].round(3)
    return results.reset_index(drop=True)

In [3]:
recommend("Inside Out")

,title,genres,year,similarity
0,Inside Out 2,"[Animation, Adventure, Comedy, Family]",2024,0.386
1,Home Sweet Home Alone,"[Family, Comedy]",2021,0.152
2,Remarkably Bright Creatures,"[Drama, Mystery]",2026,0.130
3,The Little Rascals,"[Romance, Comedy, Family]",1994,0.117
4,Ron's Gone Wrong,"[Animation, Science Fiction, Family, Comedy]",2021,0.114


In [4]:
recommend("The Godfather")

,title,genres,year,similarity
0,The Godfather Part II,"[Drama, Crime]",1974,0.461
1,The Godfather Part III,"[Crime, Drama, Thriller]",1990,0.147
2,Run All Night,"[Action, Crime, Thriller]",2015,0.110
3,Joker,"[Crime, Thriller, Drama]",2019,0.106
4,The Blind Side,[Drama],2009,0.098


In [5]:
%%writefile ../src/recommend.py
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity


def load_recommendation_data():
    df = pd.read_pickle("data/processed/movies_clustered.pkl")
    tfidf_matrix = sparse.load_npz("models/tfidf_matrix.npz")
    return df, tfidf_matrix


def recommend(title, df, tfidf_matrix, n=5):
    matches = df.index[df["title"].str.lower() == title.lower()]
    if len(matches) == 0:
        return None

    idx = matches[0]
    movie_vector = tfidf_matrix[idx]
    sims = cosine_similarity(movie_vector, tfidf_matrix).flatten()

    similar_idx = sims.argsort()[::-1]
    similar_idx = [i for i in similar_idx if i != idx][:n]

    results = df.iloc[similar_idx][["title", "genres", "year"]].copy()
    results["similarity"] = sims[similar_idx].round(3)
    return results.reset_index(drop=True)

Writing ../src/recommend.py


In [6]:
cat src/recommend.py

cat: src/recommend.py: No such file or directory
